# 📓 Update dataset with last boxscores and matches

# Please first run full pipeline until merge_clean_seasons_boxscores to have base data to work with and merge

In [1]:

import os
import pandas as pd
from datetime import datetime
from src.config import *
from src.utils import *
from src.nba_scrapping import *



In [ ]:
# ⚙️ Initialisation du run
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
current_season = '2024-25'  # À rendre dynamique si besoin plus tard


In [ ]:

# 📁 Préparation des dossiers de sortie
os.makedirs(DATA_LAST_GAMES_DIR, exist_ok=True)
os.makedirs(DATA_LAST_BOXSCORES_BATCHES_DIR, exist_ok=True)

# 🔄 Chargement des historiques si existants
hist_games_path = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)


In [ ]:

# 📥 1. Téléchargement des matchs de la saison actuelle
print("\n📥 Téléchargement des nouveaux matchs pour la saison:", current_season)
matchs_output_dir = os.path.join(DATA_LAST_GAMES_DIR, current_season)
last_games_path = download_games_for_seasons([current_season], matchs_output_dir, run_timestamp, max_retries=10)

In [ ]:

# 📊 2. Comparaison avec les données historiques pour trouver les nouveaux matchs
games_to_scrape = get_new_games(hist_games_path, last_games_path)
print(f"✅ {len(games_to_scrape)} nouveaux matchs trouvés à scraper.")

if games_to_scrape.empty:
    print("✅ Aucun nouveau match à scraper. Fin du script.")
    exit(0)

In [ ]:

# 💾 3. Merge historique + nouveaux matchs
historical_games = pd.read_csv(hist_games_path, low_memory=False, dtype={'GAME_ID': str})
all_games = pd.concat([historical_games, games_to_scrape], ignore_index=True)

save_dataframe_to_csv(all_games, DATA_LAST_GAMES_MERGED_DIR, 'last_games_merged', run_timestamp)
print("✅ Jeux de matchs fusionnés sauvegardés.")


In [ ]:

# 🏀 4. Scraping des nouveaux boxscores
print(f"--- Traitement de la saison {current_season} ---")
season_df = games_to_scrape[games_to_scrape['SEASON'] == current_season]
season_output_dir = os.path.join(DATA_LAST_BOXSCORES_BATCHES_DIR, run_timestamp, current_season)
scrape_boxscores_v3_for_games(season_df, season_output_dir,max_retries=10)

In [ ]:
print("\n✅ Mise à jour complète terminée. Données prêtes pour le feature engineering.")
